# Getting docs

In [2]:
# Numpy 2.5 docs

#https://stackoverflow.com/questions/69782728/urllib-error-httperror-http-error-403-forbidden-with-urllib-requests
import urllib.request
import zipfile

path = "raw/numpy-html"
zippath = "raw/numpy-html" + ".zip"

opener = urllib.request.build_opener()
opener.addheaders = [('User-Agent', 'MyApp/1.0')]
urllib.request.install_opener(opener)
urllib.request.urlretrieve("https://numpy.org/doc/2.5/numpy-html.zip", zippath)

#https://stackoverflow.com/questions/3451111/unzipping-files-in-python

with zipfile.ZipFile(zippath, 'r') as zip_ref:
    zip_ref.extractall(path)

# Parsing

In [3]:
from pathlib import Path
from bs4 import BeautifulSoup
from hashlib import sha256

path = Path("raw/numpy-html")

ids = []
documents = []
metadatas = []

for html_file in path.rglob("*.html"):
    
    with html_file.open("r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    title = soup.title.get_text(strip=True) if soup.title else ""

    #print(html_file)

    #print(soup)

    try:
        text = soup.find('article').text # Get article
    
        text = list(filter(None, text.split("\n"))) # Remove \n
    
        if (text[-1] == "Go BackOpen In Tab"): # Sometimes last line is this. Remove it
            text = text[:-1]
    
        text = "\n".join(text) # Add linebreks back in
    
        ids.append(sha256(text.encode('utf-8')).hexdigest())
        documents.append(text)
        metadatas.append({"file":str(html_file)})
    except:
        print("Error on file:", str(html_file)) 


Error on file: raw/numpy-html/search.html
Error on file: raw/numpy-html/lite/index.html
Error on file: raw/numpy-html/_static/webpack-macros.html
Error on file: raw/numpy-html/lite/tree/index.html
Error on file: raw/numpy-html/lite/lab/index.html
Error on file: raw/numpy-html/lite/repl/index.html
Error on file: raw/numpy-html/lite/edit/index.html
Error on file: raw/numpy-html/lite/consoles/index.html
Error on file: raw/numpy-html/lite/notebooks/index.html
Error on file: raw/numpy-html/lite/doc/tree/index.html
Error on file: raw/numpy-html/lite/doc/workspaces/index.html
Error on file: raw/numpy-html/lite/lab/tree/index.html
Error on file: raw/numpy-html/lite/lab/workspaces/index.html


In [4]:
len(ids)

2669

# Building chromadb

In [26]:
%pip install chromadb

Note: you may need to restart the kernel to use updated packages.


In [5]:
import chromadb

chroma_client = chromadb.PersistentClient("../backend/chroma.db")

In [6]:
collection = chroma_client.get_or_create_collection(name="numpy_docs")
collection.add(ids=ids, documents=documents, metadatas=metadatas)

In [11]:
results = collection.query(query_texts=["How to reshape an np.array?"])